# 00 · API check

Tests every part of the DHinfra API the project relies on. Run it top to bottom on JupyterHub with the `hisrag` kernel.

**API key:** if `DHINFRA_API_KEY` is not set yet, the first cell asks for it (hidden input) and stores it in `.env` at the repo root, readable only by you. You only need to enter it once.

Each test writes into `report`. A failing test records its error and the notebook continues.
**At the end, copy the printed JSON block and paste it back to Claude.**

In [ ]:
import json, platform, time, traceback
from importlib.metadata import version

import numpy as np
from IPython.display import Image, display
from pydantic import BaseModel

from hisrag.config import ENV_FILE, load_config, set_api_key
from hisrag.llm import DHClient

cfg = load_config()
if not cfg.api_key:
    # .env is hidden in the JupyterHub file browser; this prompt writes it for you (once).
    print(f"No API key found (looked in the environment and {ENV_FILE}, exists: {ENV_FILE.exists()})")
    set_api_key()
client = DHClient(cfg)
MODEL = cfg["llm"]["model"]

report = {
    "env": {
        "python": platform.python_version(),
        **{p: version(p) for p in ["openai", "pydantic", "numpy", "pandas", "duckdb", "pyarrow"]},
        "api_key_set": bool(cfg.api_key),
        "env_file": str(ENV_FILE),
    }
}

def run_test(name):
    """Decorator: run the test now, store its result or error in report[name]."""
    def wrap(fn):
        t0 = time.monotonic()
        try:
            report[name] = {"ok": True, **(fn() or {})}
        except Exception as exc:
            report[name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"[:800]}
            traceback.print_exc()
        report[name]["seconds"] = round(time.monotonic() - t0, 2)
        print(name, "->", json.dumps(report[name], ensure_ascii=False, indent=1)[:1500])
        return fn
    return wrap

report["env"]

In [ ]:
# Real ads from the Wiener Zeitung sample, reused by several tests below.
ADS = {
    "koechin_1850": "So wie im Kochen und jeder häuslichen Arbeit geübt, ſucht\neine Perſon als Wirthſchafterin oder Köchin in ein ſolides\nHaus einen Dienſt auf das Land. Auskunft aus Gefälligkeit\nBognergaſſe Nr. 515, in der Zwirn⸗ und Wollhandlung\nder Katharina Heyder.",
    "unterlehrer_1887": "5. Die Unterlehrer⸗, reſp. Unterlehrerinſtelle\nan der vierclaſſigen Volksſchule in Ottenſchlag,\n2. Gehaltsclaſſe, Jahresremuneration 350 fl.,\nbezw. Gehalt 500 fl.",
    "erzieherin_1867": "Als Haushälterin oder\nErzieherin zu Kinder wünſcht eine ſehr ge⸗\nbildete Dame mit Vermögen placirt zu\nwerden. (323)",
    "buchhalterin_1940s": "Bilanzbuchhalterin, erste\nKraft, sucht entsprechen-\nden Posten. Unter:\nIV. Bezirk 93.757, an\nGWG., I., Wollzeile 16.",
    "sterbefall_1883": "Biſchof Julius, Hausdiener, 21 J.,\nVI., Theobaldgaſſe 2, ſchwere Ver⸗\nletzungen. (Beſchaut in der Leichen⸗\nkammer des k. k. allgemeinen Kran⸗\nkenhauſes.)",
}
IIIF_URL = "https://iiif.onb.ac.at/images/ANNO/wrz18500326/00000029/1086,1935,941,202/full/0/default.png"

## 1 · Chat, reasoning off

In [ ]:
@run_test("chat_basic")
def _():
    r = client.chat([{"role": "user", "content": "Antworte in drei Wörtern: Was ist eine Wirthschafterin?"}],
                    use_cache=False, max_tokens=100)
    return {"content": r.content, "finish_reason": r.finish_reason, "usage": r.usage,
            "reasoning_present": bool(r.reasoning)}

## 2 · Chat, reasoning on
Checks where the reasoning text comes back and how many tokens it costs.

In [ ]:
@run_test("chat_thinking")
def _():
    r = client.chat([{"role": "user", "content": "Ist 'Unterlehrer⸗' in einer Anzeige von 1887 eine Berufsbezeichnung? Kurz."}],
                    thinking=True, max_tokens=4000, use_cache=False)
    return {"content": r.content, "finish_reason": r.finish_reason, "usage": r.usage,
            "reasoning_chars": len(r.reasoning or ""),
            "message_keys": sorted(r.raw["choices"][0]["message"].keys())}

## 3 · Structured output
Tries server-side `json_schema` (guided decoding) and the prompt-only fallback on a mini version of the step-4 task (normalizing position spellings). Both results are shown, which also gives a first look at quality.

In [ ]:
class PositionNorm(BaseModel):
    surface: str
    is_position: bool
    lemma_modern: str | None
    category: str | None
    gender: str

class PositionBatch(BaseModel):
    items: list[PositionNorm]

FORMS = ["Unterlehrer⸗", "Lehrers", "Wirthſchafterin", "eine", "Commis", "Supplentenſtelle", "Bedienter", "ſucht"]
MSGS = [
    {"role": "system", "content": "Du normalisierst Berufsbezeichnungen aus historischen Wiener Stellenanzeigen (1850–1950). "
        "Für jede Form: ist es eine Berufs-/Stellenbezeichnung, das moderne deutsche Lemma (Grundform, ohne '-stelle'), "
        "eine grobe Berufskategorie und das Geschlecht (male/female/neutral)."},
    {"role": "user", "content": "Formen:\n" + "\n".join(FORMS)},
]

for mode in ["json_schema", "prompt"]:
    @run_test(f"structured_{mode}")
    def _():
        out = client.chat_json(MSGS, PositionBatch, mode=mode, use_cache=False, max_tokens=2000)
        return {"items": [i.model_dump() for i in out.items]}

report["structured_recommended"] = ("json_schema" if report["structured_json_schema"]["ok"]
                                    else "prompt" if report["structured_prompt"]["ok"] else None)
report["structured_recommended"]

## 4 · Tool calling
A minimal agent loop with one fake tool. Tests whether Qwen calls it with valid arguments and then uses the result.

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "count_ads",
        "description": "Zählt Stellenanzeigen für eine Berufsbezeichnung in einem Jahrzehnt.",
        "parameters": {
            "type": "object",
            "properties": {
                "position": {"type": "string", "description": "Berufsbezeichnung, modernes Deutsch"},
                "decade": {"type": "integer", "description": "Jahrzehnt, z.B. 1870"},
            },
            "required": ["position", "decade"],
        },
    },
}]
FAKE_COUNTS = {1850: 120, 1860: 95, 1870: 80, 1880: 60}

def count_ads(position, decade):
    return {"position": position, "decade": decade, "n_ads": FAKE_COUNTS.get(decade, 0)}

@run_test("tool_calling")
def _():
    msgs = [{"role": "system", "content": "Du beantwortest Fragen zu historischen Stellenanzeigen. Nutze die Tools."},
            {"role": "user", "content": "Wie viele Anzeigen für Köchinnen gab es in den 1850ern und in den 1880ern?"}]
    trace = []
    for step in range(5):
        r = client.chat(msgs, tools=TOOLS, thinking=True, max_tokens=4000, use_cache=False)
        msgs.append(r.message)
        if not r.tool_calls:
            return {"steps": step + 1, "trace": trace, "final": r.content}
        for tc in r.tool_calls:
            args = json.loads(tc["function"]["arguments"])
            result = count_ads(**args)
            trace.append({"call": tc["function"]["name"], "args": args, "result": result})
            msgs.append({"role": "tool", "tool_call_id": tc["id"], "content": json.dumps(result)})
    raise RuntimeError(f"no final answer after 5 steps; trace={trace}")

## 5 · Embeddings
For every candidate model: vector size, speed for a batch of 64, and a quick check that each query is closest to the right ad. (Five ads is far too few to compare quality; that is step 8. This only checks that the setup works.)

In [ ]:
QUERIES = {
    "koechin_1850": "Frau sucht Stelle als Köchin oder Haushälterin",
    "unterlehrer_1887": "Ausschreibung Lehrerstelle Volksschule mit Gehalt",
    "erzieherin_1867": "gebildete Dame möchte als Erzieherin arbeiten",
    "buchhalterin_1940s": "Buchhalterin sucht Arbeit",
}
keys = list(ADS)
report["embeddings"] = {}
for name in cfg["embeddings"]["models"]:
    t0 = time.monotonic()
    try:
        P = client.embed([ADS[k] for k in keys], name, kind="passage")
        Q = client.embed(list(QUERIES.values()), name, kind="query")
        top = [keys[i] for i in (Q @ P.T).argmax(axis=1)]
        t1 = time.monotonic()
        client.embed([ADS[keys[i % len(keys)]] + f" {i}" for i in range(64)], name, kind="passage", batch_size=64)
        report["embeddings"][name] = {
            "ok": True, "dim": int(P.shape[1]),
            "top1_correct": f"{sum(t == q for t, q in zip(top, QUERIES))}/{len(QUERIES)}",
            "batch64_seconds": round(time.monotonic() - t1, 2),
        }
    except Exception as exc:
        report["embeddings"][name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"[:400]}
    print(name, report["embeddings"][name])

## 6 · Parallel throughput
30 short requests on 16 threads, reasoning off. Tests the rate limiter and the server's concurrency.

In [ ]:
@run_test("throughput")
def _():
    prompts = [f"Nenne ein Synonym für 'Dienstbote' (Variante {i}). Nur ein Wort." for i in range(30)]
    def ask(p):
        return client.chat([{"role": "user", "content": p}], max_tokens=20, use_cache=False)
    t0 = time.monotonic()
    out = client.map(ask, prompts, max_workers=16, return_exceptions=True)
    elapsed = time.monotonic() - t0
    errors = [f"{type(o).__name__}: {o}"[:200] for o in out if isinstance(o, Exception)]
    return {"requests": len(prompts), "wall_seconds": round(elapsed, 2),
            "requests_per_second": round(len(prompts) / elapsed, 2), "errors": errors[:5], "n_errors": len(errors)}

## 7 · Response cache
The same request twice: the second answer must come from the local cache without calling the API.

In [ ]:
@run_test("cache")
def _():
    m = [{"role": "user", "content": "Cache-Test: antworte mit 'ok'."}]
    a = client.chat(m, max_tokens=10)
    b = client.chat(m, max_tokens=10)
    return {"first_from_cache": a.from_cache, "second_from_cache": b.from_cache, "same": a.content == b.content}

## 8 · IIIF access and vision
Loads one clipping from the ONB IIIF server and asks Qwen to transcribe it. This is a first test of whether reading the image could help with garbled OCR later.

In [ ]:
import base64, requests

@run_test("iiif_vision")
def _():
    resp = requests.get(IIIF_URL, timeout=30)
    resp.raise_for_status()
    display(Image(data=resp.content))
    b64 = base64.b64encode(resp.content).decode()
    r = client.chat([{"role": "user", "content": [
        {"type": "text", "text": "Transkribiere diese Zeitungsanzeige (Fraktur) buchstabengetreu."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
    ]}], max_tokens=600, use_cache=False)
    return {"image_bytes": len(resp.content), "transcription": r.content, "reference": ADS["koechin_1850"],
            "usage": r.usage}

## Report
Copy everything between the lines and paste it back.

In [ ]:
report["usage_by_job"] = client.usage.summary()
print("-" * 80)
print(json.dumps(report, ensure_ascii=False, indent=1))
print("-" * 80)